# 🤖 Exercice 3 — Mini-Pipeline RAG Complet

**Durée estimée : 35 minutes**

## Objectif

Assembler les briques des exercices 1 et 2 en un **pipeline RAG fonctionnel**
qui, pour une requête utilisateur :

1. **Classifie l'intention** (intent classifier)
2. **Recherche** les films pertinents (embeddings + cosinus)
3. **Construit le prompt** (system prompt + contexte + question)
4. **Appelle un LLM** pour générer la réponse

## Deux modes selon votre machine

Ce notebook **détecte automatiquement** votre configuration :

| Config | Mode | Expérience |
|--------|------|------------|
| **CPU uniquement** | 🔵 Simulé | Le pipeline tourne, le LLM est remplacé par un générateur basique |
| **GPU NVIDIA (6+ Go VRAM)** | 🟢 Ollama | Un vrai LLM local génère les réponses en temps réel |

> 💡 **Astuce en groupe** : si un seul PC a un GPU, les autres peuvent
> s'y connecter en réseau (instructions dans la partie setup).

---
## Partie 1 — Détection de votre configuration

In [16]:
import shutil
import subprocess
import urllib.request
import os

def detect_gpu() -> dict | None:
    """Détecte la présence et les specs d'un GPU NVIDIA.

    Returns:
        Dict avec les infos GPU (name, vram_mb) ou None.
    """
    try:
        import torch
        if torch.cuda.is_available():
            name = torch.cuda.get_device_name(0)
            vram_mb = torch.cuda.get_device_properties(0).total_mem // (1024 * 1024)
            return {"name": name, "vram_mb": vram_mb}
    except ImportError:
        pass

    if shutil.which("nvidia-smi"):
        try:
            out = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=name,memory.total",
                 "--format=csv,noheader,nounits"],
                text=True,
            ).strip()
            parts = out.split(", ")
            if len(parts) == 2:
                return {"name": parts[0], "vram_mb": int(parts[1])}
        except Exception:
            pass
    return None


def detect_ollama() -> bool:
    """Vérifie si Ollama est installé et accessible.

    Returns:
        True si Ollama répond sur localhost.
    """
    try:
        req = urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3)
        return req.status == 200
    except Exception:
        return False


# ── Détection ──
gpu_info = detect_gpu()
ollama_ready = detect_ollama()

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024**3)
except ImportError:
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / (1024**3)

print("╔══════════════════════════════════════════╗")
print("║       DÉTECTION CONFIGURATION            ║")
print("╠══════════════════════════════════════════╣")
print(f"║  RAM totale    : {ram_gb:.0f} Go")
if gpu_info:
    print(f"║  GPU détecté   : {gpu_info['name']}")
    print(f"║  VRAM          : {gpu_info['vram_mb']} Mo ({gpu_info['vram_mb'] // 1024} Go)")
else:
    print("║  GPU détecté   : ❌ Aucun GPU NVIDIA")
print(f"║  Ollama actif  : {'✅ Oui' if ollama_ready else '❌ Non'}")
print("╚══════════════════════════════════════════╝")

╔══════════════════════════════════════════╗
║       DÉTECTION CONFIGURATION            ║
╠══════════════════════════════════════════╣
║  RAM totale    : 31 Go
║  GPU détecté   : NVIDIA RTX 2000 Ada Generation Laptop GPU
║  VRAM          : 8188 Mo (7 Go)
║  Ollama actif  : ✅ Oui
╚══════════════════════════════════════════╝


In [17]:
# ── Choix automatique du mode ──
LLM_MODE = "simulated"
LLM_API_URL = ""
LLM_MODEL = ""
RECOMMENDED_MODEL = ""

if gpu_info and gpu_info["vram_mb"] >= 6000:
    RECOMMENDED_MODEL = "qwen2.5:7b" if gpu_info["vram_mb"] >= 8000 else "qwen2.5:3b"
    if ollama_ready:
        LLM_MODE = "ollama"
        LLM_API_URL = "http://localhost:11434/v1/chat/completions"
        LLM_MODEL = RECOMMENDED_MODEL
elif ollama_ready:
    LLM_MODE = "ollama"
    LLM_API_URL = "http://localhost:11434/v1/chat/completions"
    LLM_MODEL = "qwen2.5:3b"
    RECOMMENDED_MODEL = "qwen2.5:3b"

if LLM_MODE == "ollama":
    print(f"🟢 MODE LLM RÉEL — Ollama + {LLM_MODEL}")
    print(f"   Les réponses seront générées par un vrai modèle de langage.")
elif gpu_info and gpu_info["vram_mb"] >= 6000:
    print(f"🟡 GPU détecté mais Ollama non installé.")
    print(f"   → Suivez les instructions ci-dessous pour installer Ollama")
    print(f"   → Modèle recommandé : {RECOMMENDED_MODEL}")
else:
    print(f"🔵 MODE SIMULÉ — Pas de GPU ou Ollama non disponible")
    print(f"   Le pipeline complet fonctionne, le LLM est remplacé par")
    print(f"   un générateur basique. L'apprentissage est identique.")

🟢 MODE LLM RÉEL — Ollama + qwen2.5:7b
   Les réponses seront générées par un vrai modèle de langage.


---
### ⚙️ Installation Ollama (GPU disponible ou mode réseau)

> **Passez cette section si vous êtes en mode simulé 🔵**

[Ollama](https://ollama.com) permet de faire tourner des LLM en local
en une seule commande.

**Installation :**

| OS | Commande |
|----|----------|
| **Windows** | Télécharger sur [ollama.com/download](https://ollama.com/download) |
| **Linux** | `curl -fsSL https://ollama.com/install.sh \| sh` |
| **Mac** | `brew install ollama` |

**Démarrage (dans un terminal séparé) :**

```bash
ollama serve
```

**Télécharger le modèle (dans un autre terminal) :**

```bash
ollama pull qwen2.5:3b    # GPU 6 Go VRAM (~2 Go download)
ollama pull qwen2.5:7b    # GPU 8+ Go VRAM (~4.7 Go download)
```

Puis **ré-exécutez les 2 cellules de détection** ci-dessus.

---

### 🌐 Mode réseau — partager le LLM avec le groupe

Si un seul PC a un GPU, les autres peuvent s'y connecter.

**Sur le PC avec GPU** ("serveur") :
```bash
OLLAMA_HOST=0.0.0.0:11434 ollama serve
```
(Windows : définir la variable d'environnement `OLLAMA_HOST=0.0.0.0:11434` avant de lancer)

**Sur les autres PCs** ("clients") — modifiez la cellule ci-dessous :

In [ ]:
# ══════════════════════════════════════════════════
# MODE RÉSEAU — Décommentez et modifiez si vous
# utilisez le GPU d'un autre PC du groupe
# ══════════════════════════════════════════════════

# ADRESSE_PC_GPU = "192.168.1.42"  # ← Remplacez par l'IP du PC avec GPU
#
# LLM_MODE = "ollama"
# LLM_API_URL = f"http://{ADRESSE_PC_GPU}:11434/v1/chat/completions"
# LLM_MODEL = "qwen2.5:7b"
# print(f"🌐 Mode réseau activé → {LLM_API_URL}")

---
## Partie 2 — Rechargement des composants

In [18]:
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.svm import LinearSVC
from sklearn.metrics.pairwise import cosine_similarity

embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Modèle d'embedding chargé")

with open("horror_movies_dataset.json", encoding="utf-8") as f:
    movies = json.load(f)
overviews = [m["overview"] for m in movies]
movie_embeddings = embed_model.encode(overviews, show_progress_bar=True)
print(f"✅ {len(movies)} films encodés ({movie_embeddings.shape})")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modèle d'embedding chargé


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ 50 films encodés ((50, 384))


In [19]:
# ── Intent classifier (repris du notebook 02) ──
TRAINING_DATA: list[dict] = [
    {"text": "Recommande-moi un bon film d'horreur", "intent": "recommendation"},
    {"text": "Je cherche un film de zombies", "intent": "recommendation"},
    {"text": "Un film effrayant pour ce soir", "intent": "recommendation"},
    {"text": "Quel film d'horreur regarder ?", "intent": "recommendation"},
    {"text": "Suggère-moi un film avec des fantômes", "intent": "recommendation"},
    {"text": "Je veux un film qui fait vraiment peur", "intent": "recommendation"},
    {"text": "Un bon slasher à voir ?", "intent": "recommendation"},
    {"text": "Film d'horreur psychologique récent", "intent": "recommendation"},
    {"text": "Quelque chose de flippant avec des démons", "intent": "recommendation"},
    {"text": "Un film de maison hantée", "intent": "recommendation"},
    {"text": "Je veux voir un film de vampire", "intent": "recommendation"},
    {"text": "Un film d'horreur français", "intent": "recommendation"},
    {"text": "Bonjour", "intent": "greeting"},
    {"text": "Salut !", "intent": "greeting"},
    {"text": "Hello", "intent": "greeting"},
    {"text": "Hey", "intent": "greeting"},
    {"text": "Coucou", "intent": "greeting"},
    {"text": "Bonsoir", "intent": "greeting"},
    {"text": "Bonjour, comment ça va ?", "intent": "greeting"},
    {"text": "Salut, tu peux m'aider ?", "intent": "greeting"},
    {"text": "Re", "intent": "greeting"},
    {"text": "Yo", "intent": "greeting"},
    {"text": "Qui a réalisé Shining ?", "intent": "question_info"},
    {"text": "En quelle année est sorti L'Exorciste ?", "intent": "question_info"},
    {"text": "Quelle est la note de Hereditary ?", "intent": "question_info"},
    {"text": "De quoi parle le film Get Out ?", "intent": "question_info"},
    {"text": "Combien a coûté le film Alien ?", "intent": "question_info"},
    {"text": "Qui joue dans Scream ?", "intent": "question_info"},
    {"text": "C'est quoi le synopsis de Midsommar ?", "intent": "question_info"},
    {"text": "Le réalisateur de Conjuring", "intent": "question_info"},
    {"text": "Quelle est la durée de Saw ?", "intent": "question_info"},
    {"text": "La date de sortie de Nope", "intent": "question_info"},
    {"text": "Quelle est la météo aujourd'hui ?", "intent": "out_of_scope"},
    {"text": "Raconte-moi une blague", "intent": "out_of_scope"},
    {"text": "Quel est le cours du Bitcoin ?", "intent": "out_of_scope"},
    {"text": "Comment faire une omelette ?", "intent": "out_of_scope"},
    {"text": "Qui est le président de la France ?", "intent": "out_of_scope"},
    {"text": "Aide-moi avec mon code Python", "intent": "out_of_scope"},
    {"text": "Traduis cette phrase en anglais", "intent": "out_of_scope"},
    {"text": "Quel est le score du match ?", "intent": "out_of_scope"},
    {"text": "Réserve-moi un restaurant", "intent": "out_of_scope"},
    {"text": "Comment aller à la gare ?", "intent": "out_of_scope"},
]

train_texts = [d["text"] for d in TRAINING_DATA]
train_labels = [d["intent"] for d in TRAINING_DATA]
train_embs = embed_model.encode(train_texts, show_progress_bar=False)

intent_classifier = LinearSVC(random_state=42, max_iter=1000)
intent_classifier.fit(train_embs, train_labels)
print("✅ Intent classifier entraîné")

✅ Intent classifier entraîné


---
## Partie 3 — System prompts par intent

In [20]:
SYSTEM_PROMPTS: dict[str, str] = {
    "recommendation": (
        "Tu es HorrorBot, un expert en films d'horreur. "
        "Recommande des films en te basant UNIQUEMENT sur le contexte fourni. "
        "Ne cite jamais un film qui n'est pas dans le contexte. "
        "Si aucun film ne correspond, dis-le honnêtement. "
        "Réponds en français, de manière enthousiaste mais concise."
    ),
    "greeting": (
        "Tu es HorrorBot, un chatbot spécialisé dans les films d'horreur. "
        "Réponds brièvement et chaleureusement à la salutation. "
        "Invite l'utilisateur à te poser une question sur les films d'horreur. "
        "Réponds en français."
    ),
    "question_info": (
        "Tu es HorrorBot, un expert en films d'horreur. "
        "Réponds factuellement à la question en utilisant UNIQUEMENT "
        "le contexte fourni. Si l'information n'est pas disponible "
        "dans le contexte, dis-le clairement. Réponds en français."
    ),
    "out_of_scope": (
        "Tu es HorrorBot, un chatbot spécialisé dans les films d'horreur. "
        "Explique poliment que tu ne peux aider que sur les films d'horreur. "
        "Donne un exemple de question que l'utilisateur pourrait poser. "
        "Réponds en français."
    ),
}

INTENTS_REQUIRING_RAG = {"recommendation", "question_info"}

print("System prompts configurés :")
for intent, prompt in SYSTEM_PROMPTS.items():
    rag = "🔍 RAG" if intent in INTENTS_REQUIRING_RAG else "💬 Direct"
    print(f"  {rag}  {intent:<20} → {prompt[:60]}...")

System prompts configurés :
  🔍 RAG  recommendation       → Tu es HorrorBot, un expert en films d'horreur. Recommande de...
  💬 Direct  greeting             → Tu es HorrorBot, un chatbot spécialisé dans les films d'horr...
  🔍 RAG  question_info        → Tu es HorrorBot, un expert en films d'horreur. Réponds factu...
  💬 Direct  out_of_scope         → Tu es HorrorBot, un chatbot spécialisé dans les films d'horr...


---
## Partie 4 — Fonctions du pipeline

In [21]:
def classify_intent(query: str) -> str:
    """Classifie l'intention de la requête utilisateur.

    Args:
        query: Message de l'utilisateur.

    Returns:
        L'intent détecté parmi les 4 catégories.
    """
    query_emb = embed_model.encode([query])
    return intent_classifier.predict(query_emb)[0]


def search_relevant_movies(query: str, top_k: int = 3) -> list[dict]:
    """Recherche les films les plus pertinents par similarité cosinus.

    Args:
        query: Requête de l'utilisateur.
        top_k: Nombre de films à retourner.

    Returns:
        Liste des films avec score et métadonnées.
    """
    query_vec = embed_model.encode([query])
    similarities = cosine_similarity(query_vec, movie_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "title": movies[idx]["title"],
            "overview": movies[idx]["overview"],
            "vote_average": movies[idx]["vote_average"],
            "release_date": movies[idx].get("release_date", "N/A"),
            "score": float(similarities[idx]),
        })
    return results


def build_context(relevant_movies: list[dict]) -> str:
    """Formate les films en texte de contexte pour le prompt.

    Args:
        relevant_movies: Films retournés par la recherche.

    Returns:
        Texte formaté pour injection dans le prompt.
    """
    if not relevant_movies:
        return ""
    lines = ["--- CONTEXTE (films trouvés dans la base de données) ---"]
    for m in relevant_movies:
        year = m["release_date"][:4] if m["release_date"] else "N/A"
        lines.append(f"Film: {m['title']} ({year}) | Note: {m['vote_average']}/10")
        lines.append(f"Synopsis: {m['overview']}")
        lines.append("")
    lines.append("--- FIN DU CONTEXTE ---")
    return "\n".join(lines)


def build_messages(
    user_query: str, intent: str, context: str,
) -> list[dict[str, str]]:
    """Construit les messages pour l'API LLM (format ChatML).

    Args:
        user_query: Message de l'utilisateur.
        intent: Intent détecté.
        context: Contexte RAG formaté.

    Returns:
        Liste de messages [{role, content}, ...].
    """
    system_prompt = SYSTEM_PROMPTS[intent]
    system_content = f"{system_prompt}\n\n{context}" if context else system_prompt
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_query},
    ]


print("✅ Fonctions du pipeline définies")

✅ Fonctions du pipeline définies


---
## Partie 5 — Le moteur LLM

- **🟢 Ollama** : appel HTTP au serveur local (vrai LLM)
- **🔵 Simulé** : réponse construite à partir du contexte RAG

In [22]:
import requests


def call_llm_ollama(messages: list[dict[str, str]]) -> str:
    """Appelle Ollama via l'API compatible OpenAI.

    Args:
        messages: Messages au format ChatML.

    Returns:
        Réponse générée par le LLM.
    """
    payload = {
        "model": LLM_MODEL,
        "messages": messages,
        "temperature": 0.7,
        "max_tokens": 300,
        "stream": False,
    }
    try:
        response = requests.post(LLM_API_URL, json=payload, timeout=120)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]
    except requests.exceptions.ConnectionError:
        return (
            "❌ Impossible de se connecter à Ollama.\n"
            "   Vérifiez que 'ollama serve' tourne dans un terminal séparé."
        )
    except Exception as e:
        return f"❌ Erreur LLM : {e}"


def call_llm_simulated(messages: list[dict[str, str]]) -> str:
    """Simule un appel LLM sans GPU.

    Args:
        messages: Messages au format ChatML.

    Returns:
        Réponse simulée montrant la mécanique du pipeline.
    """
    system = messages[0]["content"]

    if "CONTEXTE" in system:
        film_lines = [l for l in system.split("\n") if l.startswith("Film:")]
        if film_lines:
            films = "\n".join(f"  • {l.replace('Film: ', '')}" for l in film_lines[:3])
            return (
                f"[MODE SIMULÉ — en production, Qwen2.5 génère cette réponse]\n\n"
                f"Films pertinents trouvés par le RAG :\n{films}\n\n"
                f"Le LLM rédigerait une réponse naturelle à partir de ces synopsis."
            )

    if "salutation" in system.lower() or "chaleureusement" in system.lower():
        return (
            "[MODE SIMULÉ]\n\n"
            "Bonjour ! 👋 Je suis HorrorBot, ton expert en films d'horreur. "
            "Qu'est-ce qui te ferait plaisir ?"
        )

    return (
        "[MODE SIMULÉ]\n\n"
        "Je suis HorrorBot, spécialisé dans les films d'horreur. "
        "Essaie : « Recommande-moi un bon film d'horreur ! »"
    )


def call_llm(messages: list[dict[str, str]]) -> str:
    """Appelle le LLM selon le mode détecté.

    Args:
        messages: Messages au format ChatML.

    Returns:
        Réponse du LLM.
    """
    if LLM_MODE == "ollama":
        return call_llm_ollama(messages)
    return call_llm_simulated(messages)


# ── Vérification ──
if LLM_MODE == "ollama":
    print(f"🟢 LLM réel via Ollama — modèle : {LLM_MODEL}")
    try:
        resp = requests.get("http://localhost:11434/api/tags", timeout=5)
        models = [m["name"] for m in resp.json().get("models", [])]
        base = LLM_MODEL.split(":")[0]
        if any(base in m for m in models):
            print(f"   ✅ Modèle '{LLM_MODEL}' disponible")
        else:
            print(f"   ⚠️  Modèle non trouvé → ollama pull {LLM_MODEL}")
            print(f"   Modèles dispo : {', '.join(models) or 'aucun'}")
    except Exception:
        pass
else:
    print("🔵 Mode simulé — le pipeline fonctionne sans LLM réel")

🟢 LLM réel via Ollama — modèle : qwen2.5:7b
   ✅ Modèle 'qwen2.5:7b' disponible


---
## Partie 6 — Le pipeline complet

In [23]:
def rag_pipeline(user_query: str, verbose: bool = True, top_k: int = 3) -> str:
    """Pipeline RAG complet : intent → search → prompt → LLM → réponse.

    Args:
        user_query: Message de l'utilisateur.
        verbose: Affiche les étapes intermédiaires.
        top_k: Nombre de films pour la recherche RAG.

    Returns:
        Réponse générée par le LLM.
    """
    if verbose:
        tag = "🟢 Ollama" if LLM_MODE == "ollama" else "🔵 Simulé"
        print(f'{tag}  Requête : "{user_query}"')
        print("─" * 60)

    # 1. Intent
    intent = classify_intent(user_query)
    if verbose:
        print(f"🎯 Intent  : {intent}")

    # 2. RAG
    context = ""
    if intent in INTENTS_REQUIRING_RAG:
        relevant = search_relevant_movies(user_query, top_k=top_k)
        context = build_context(relevant)
        if verbose:
            print("🔍 Films trouvés :")
            for r in relevant:
                print(f"   {r['score']:.3f}  ★{r['vote_average']}  {r['title']}")
    elif verbose:
        print(f"⏭️  Pas de recherche RAG (intent: {intent})")

    # 3. Prompt
    messages = build_messages(user_query, intent, context)
    if verbose:
        print(f"📝 Prompt  : {SYSTEM_PROMPTS[intent][:60]}...")
        print("─" * 60)
        print("🤖 Réponse :\n")

    # 4. LLM
    response = call_llm(messages)
    print(response)
    return response


print("✅ Pipeline RAG assemblé !")

✅ Pipeline RAG assemblé !


---
## Partie 7 — Test du pipeline

In [24]:
# Test 1 : Recommandation (RAG activé)
print("=" * 60)
rag_pipeline("Recommande-moi un bon film avec des fantômes")

🟢 Ollama  Requête : "Recommande-moi un bon film avec des fantômes"
────────────────────────────────────────────────────────────
🎯 Intent  : recommendation
🔍 Films trouvés :
   0.473  ★7.294  Conjuring 2 : Le Cas Enfield
   0.396  ★7.425  Scream
   0.378  ★7.12  Constantine
📝 Prompt  : Tu es HorrorBot, un expert en films d'horreur. Recommande de...
────────────────────────────────────────────────────────────
🤖 Réponse :

Conjuring 2 : Le Cas Enfield devrait te plaire ! C'est un excellent thriller paranormal qui explore les expériences réelles de la famille Warren face à une maison hantée terrifiante.


"Conjuring 2 : Le Cas Enfield devrait te plaire ! C'est un excellent thriller paranormal qui explore les expériences réelles de la famille Warren face à une maison hantée terrifiante."

In [25]:
# Test 2 : Greeting (pas de RAG)
print("\n" + "=" * 60)
rag_pipeline("Salut !")


🟢 Ollama  Requête : "Salut !"
────────────────────────────────────────────────────────────
🎯 Intent  : greeting
⏭️  Pas de recherche RAG (intent: greeting)
📝 Prompt  : Tu es HorrorBot, un chatbot spécialisé dans les films d'horr...
────────────────────────────────────────────────────────────
🤖 Réponse :

Bonjour ! Bienvenue ! Quel film d'horreur te tente aujourd'hui ? Peux-tu me parler de ton préféré ou simplement chercher des recommandations ?


"Bonjour ! Bienvenue ! Quel film d'horreur te tente aujourd'hui ? Peux-tu me parler de ton préféré ou simplement chercher des recommandations ?"

In [26]:
# Test 3 : Question info (RAG activé)
print("\n" + "=" * 60)
rag_pipeline("De quoi parle le film Hereditary ?")


🟢 Ollama  Requête : "De quoi parle le film Hereditary ?"
────────────────────────────────────────────────────────────
🎯 Intent  : question_info
🔍 Films trouvés :
   0.481  ★7.294  Conjuring 2 : Le Cas Enfield
   0.419  ★7.539  Conjuring : Les Dossiers Warren
   0.370  ★7.425  Scream
📝 Prompt  : Tu es HorrorBot, un expert en films d'horreur. Réponds factu...
────────────────────────────────────────────────────────────
🤖 Réponse :

D'après le contexte fourni, aucun film intitulé "Hereditary" n'est mentionné dans la liste des films de la base de données. Les films listés sont Conjuring 2 : Le Cas Enfield, Conjuring : Les Dossiers Warren et Scream. Pour obtenir des informations sur Hereditary, nous aurions besoin d'un contexte supplémentaire ou différent.


'D\'après le contexte fourni, aucun film intitulé "Hereditary" n\'est mentionné dans la liste des films de la base de données. Les films listés sont Conjuring 2 : Le Cas Enfield, Conjuring : Les Dossiers Warren et Scream. Pour obtenir des informations sur Hereditary, nous aurions besoin d\'un contexte supplémentaire ou différent.'

In [27]:
# Test 4 : Out of scope (pas de RAG)
print("\n" + "=" * 60)
rag_pipeline("Quelle est la capitale du Japon ?")


🟢 Ollama  Requête : "Quelle est la capitale du Japon ?"
────────────────────────────────────────────────────────────
🎯 Intent  : out_of_scope
⏭️  Pas de recherche RAG (intent: out_of_scope)
📝 Prompt  : Tu es HorrorBot, un chatbot spécialisé dans les films d'horr...
────────────────────────────────────────────────────────────
🤖 Réponse :

Je suis désolé, mais je peux vous aider avec des questions liées aux films d'horreur. Si vous avez des questions sur ce sujet, n'hésitez pas à me demander ! Par exemple, vous pourriez me poser : "Quel est le meilleur film d'horreur selon vous ?"


'Je suis désolé, mais je peux vous aider avec des questions liées aux films d\'horreur. Si vous avez des questions sur ce sujet, n\'hésitez pas à me demander ! Par exemple, vous pourriez me poser : "Quel est le meilleur film d\'horreur selon vous ?"'

---
## Partie 8 — À vous de jouer !

### Exercice 8.1 — Conversation libre

In [28]:
# Exercice 8.1 — Testez vos propres requêtes
mes_requetes = [
    "Hey !",
    "Je veux un film de vampire bien flippant",
    "C'est quoi les meilleurs films d'horreur psychologique ?",
    "Donne-moi la recette des crêpes",
    # Ajoutez les vôtres :
]

for q in mes_requetes:
    print("\n" + "=" * 60)
    rag_pipeline(q)


🟢 Ollama  Requête : "Hey !"
────────────────────────────────────────────────────────────
🎯 Intent  : greeting
⏭️  Pas de recherche RAG (intent: greeting)
📝 Prompt  : Tu es HorrorBot, un chatbot spécialisé dans les films d'horr...
────────────────────────────────────────────────────────────
🤖 Réponse :

Salut ! Content de te parler de films d'horreur. Quel est ton film préféré dans ce genre ou as-tu envie de découvrir quelque chose de new et effrayant ?

🟢 Ollama  Requête : "Je veux un film de vampire bien flippant"
────────────────────────────────────────────────────────────
🎯 Intent  : recommendation
🔍 Films trouvés :
   0.560  ★6.82  Blade
   0.414  ★7.294  Conjuring 2 : Le Cas Enfield
   0.376  ★7.539  Conjuring : Les Dossiers Warren
📝 Prompt  : Tu es HorrorBot, un expert en films d'horreur. Recommande de...
────────────────────────────────────────────────────────────
🤖 Réponse :

Blade (1998) pourrait être une excellente choix ! Malgré sa note moyenne, il offre une combinaison int

### Exercice 8.2 — Modifier les system prompts

Changez le style des réponses. Idées : expert sarcastique,
bibliothécaire passionné, critique de cinéma snob...

> 💡 **En mode Ollama 🟢**, les changements modifient *réellement*
> le comportement du LLM. Essayez des prompts très différents !

In [29]:
# Exercice 8.2 — Personnalisez le system prompt
SYSTEM_PROMPTS["recommendation"] = (
    # TODO : Écrivez votre propre system prompt ici !
    "Tu es HorrorBot, un expert en films d'horreur. "
    "Recommande des films en te basant UNIQUEMENT sur le contexte fourni. "
    "Réponds en français, de manière enthousiaste mais concise."
)

print("=" * 60)
rag_pipeline("Un bon film de zombies ?")

🟢 Ollama  Requête : "Un bon film de zombies ?"
────────────────────────────────────────────────────────────
🎯 Intent  : recommendation
🔍 Films trouvés :
   0.464  ★7.26  Bienvenue à Zombieland
   0.441  ★6.82  Blade
   0.408  ★7.425  Scream
📝 Prompt  : Tu es HorrorBot, un expert en films d'horreur. Recommande de...
────────────────────────────────────────────────────────────
🤖 Réponse :

Bienvenue à Zombieland est définitivement un excellent choix ! Avec sa combinaison de comédie et d'action, ce film propose une approche originale des zombies. C'est l'histoire parfaitement dosée entre survie haletante et moments de légèreté qui en font un classique du genre.


"Bienvenue à Zombieland est définitivement un excellent choix ! Avec sa combinaison de comédie et d'action, ce film propose une approche originale des zombies. C'est l'histoire parfaitement dosée entre survie haletante et moments de légèreté qui en font un classique du genre."

### Exercice 8.3 — Impact du Top-K

In [ ]:
# Exercice 8.3 — Comparez top_k=1 vs top_k=5
query = "Film d'horreur avec des enfants possédés"

print("=== TOP-K = 1 ===")
rag_pipeline(query, top_k=1)

print("\n\n=== TOP-K = 5 ===")
rag_pipeline(query, top_k=5)

### Exercice 8.4 — Inspecter le prompt complet envoyé au LLM

In [ ]:
# Exercice 8.4 — Voir exactement ce que le LLM reçoit
query = "Un film de fantômes japonais"
intent = classify_intent(query)
relevant = search_relevant_movies(query, top_k=3)
context = build_context(relevant)
messages = build_messages(query, intent, context)

print("╔══════════════════════════════════════════════╗")
print("║  PROMPT COMPLET ENVOYÉ AU LLM               ║")
print("╠══════════════════════════════════════════════╣")
for msg in messages:
    print(f"\n── {msg['role'].upper()} ──")
    print(msg["content"])
print("\n╚══════════════════════════════════════════════╝")

total_chars = sum(len(m["content"]) for m in messages)
print(f"\n📊 ~{total_chars} caractères ≈ {total_chars // 4} tokens")

---
## Récapitulatif

Vous avez assemblé un **pipeline RAG complet et fonctionnel** :

```
"Recommande-moi un film de zombies"
         │
         ▼
┌─────────────────────────┐
│  1. Intent Classifier    │  → recommendation
│     (Embeddings + SVM)   │
└────────┬────────────────┘
         ▼
┌─────────────────────────┐
│  2. Recherche RAG        │  → Top-K films par cosinus
└────────┬────────────────┘
         ▼
┌─────────────────────────┐
│  3. Prompt augmenté      │  → system + contexte + question
└────────┬────────────────┘
         ▼
┌─────────────────────────┐
│  4. LLM génère           │  → 🟢 Ollama ou 🔵 Simulé
└─────────────────────────┘
```

### Tutoriel vs Production

| Composant | Ce tutoriel | HorrorBot production |
|-----------|-------------|---------------------|
| Intent classifier | SVM + MiniLM | DeBERTa-v3 fine-tuné |
| Stockage embeddings | Numpy en mémoire | PostgreSQL + pgvector |
| Recherche | sklearn cosine_similarity | SQL vectoriel pgvector |
| LLM | Ollama / simulé | llama-cpp-python (Qwen2.5-7B) |
| API | Fonction Python | FastAPI + JWT + rate limiting |
| Monitoring | Aucun | Prometheus + Grafana |

### Bravo ! 🎉